In [1]:
import sys
import os

sys.path.append("..")

from pathlib import Path
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import json
from pydantic import BaseModel
import plotly.express as px

In [2]:
path_train = Path("../data/states/sequence_train")
path_valid = Path("../data/states/sequence_valid")
path_test = Path("../data/states/sequence_test")

In [3]:
class HMMState(BaseModel):
    cable: int
    state: int
    norm_state: float
    avg_tension: float
    avg_radius: float
    avg_angle: float


class SampleState(BaseModel):
    sampled_at: str
    num_states: int
    c0: HMMState
    c1: HMMState
    c2: HMMState
    c3: HMMState

In [4]:
def load_raw_states(path: Path) -> list[dict]:
    data = []
    for file in os.listdir(path):
        if not file.endswith(".json"):
            continue
        with open(path / file, "r") as f:
            states = json.load(f)
        for cable in [0, 1, 2, 3]:
            content = states[f"c{cable}"]
            content["sampled_at"] = states["sampled_at"]
            content["num_states"] = states["num_states"]
            data.append(content)
    return data

In [5]:
train_states = load_raw_states(path_train)
valid_states = load_raw_states(path_valid)
test_states = load_raw_states(path_test)

In [6]:
train_states[0]

{'cable': 0,
 'state': 0,
 'norm_state': 0.0,
 'avg_tension': 2666.9319788359944,
 'avg_radius': 0.06964235720315524,
 'avg_angle': 0.018077383290173613,
 'sampled_at': '2023-04-27T00:00:00',
 'num_states': 6}

In [167]:
train_df = pd.DataFrame(train_states).sort_values("sampled_at")
valid_df = pd.DataFrame(valid_states).sort_values("sampled_at")
test_df = pd.DataFrame(test_states).sort_values("sampled_at")

In [168]:
train_df["elapsed_days"] = pd.to_datetime(train_df["sampled_at"]) - pd.to_datetime(
    train_df["sampled_at"].min()
)
train_df["elapsed_days"] = train_df["elapsed_days"].dt.days

# ---- 
valid_df["elapsed_days"] = pd.to_datetime(valid_df["sampled_at"]) - pd.to_datetime(
    valid_df["sampled_at"].min()
)
valid_df["elapsed_days"] = valid_df["elapsed_days"].dt.days

# ----
test_df["elapsed_days"] = pd.to_datetime(test_df["sampled_at"]) - pd.to_datetime(
    test_df["sampled_at"].min()
)
test_df["elapsed_days"] = test_df["elapsed_days"].dt.days

In [169]:
train_first_tension = train_df.groupby("cable")["avg_tension"].max().to_dict()
valid_first_tension = valid_df.groupby("cable")["avg_tension"].max().to_dict()
test_first_tension = test_df.groupby("cable")["avg_tension"].max().to_dict()


In [170]:
train_df["avg_tension_drop"] = train_df.apply(lambda x: train_first_tension[x["cable"]] - x["avg_tension"], axis=1)
valid_df["avg_tension_drop"] = valid_df.apply(lambda x: valid_first_tension[x["cable"]] - x["avg_tension"], axis=1)
test_df["avg_tension_drop"] = test_df.apply(lambda x: test_first_tension[x["cable"]] - x["avg_tension"], axis=1)

In [171]:
train_df_per_cable = {
    cable: df_cable.reset_index(drop=True).sort_values("sampled_at")
    for cable, df_cable in train_df.groupby("cable")
}

valid_df_per_cable = {
    cable: df_cable.reset_index(drop=True).sort_values("sampled_at")
    for cable, df_cable in valid_df.groupby("cable")
}

test_df_per_cable = {
    cable: df_cable.reset_index(drop=True).sort_values("sampled_at")
    for cable, df_cable in test_df.groupby("cable")
}

In [172]:
train_state_first_day = (
    train_df.groupby(["cable", "state"])["elapsed_days"].min().to_dict()
)
valid_state_first_day = (
    valid_df.groupby(["cable", "state"])["elapsed_days"].min().to_dict()
)
test_state_first_day = (
    test_df.groupby(["cable", "state"])["elapsed_days"].min().to_dict()
)

In [173]:
train_df["avg_angle_deg"] = np.degrees(train_df["avg_angle"])
train_df["state"] = train_df["state"].astype(str)
train_df["cable"] = train_df["cable"].astype(str)
train_df["dataset"] = "train"

valid_df["avg_angle_deg"] = np.degrees(valid_df["avg_angle"])
valid_df["state"] = valid_df["state"].astype(str)
valid_df["cable"] = valid_df["cable"].astype(str)
valid_df["dataset"] = "valid"

test_df["avg_angle_deg"] = np.degrees(test_df["avg_angle"])
test_df["state"] = test_df["state"].astype(str)
test_df["cable"] = test_df["cable"].astype(str)
test_df["dataset"] = "test"

In [174]:
full_df = pd.concat([train_df, valid_df, test_df]).reset_index(drop=True)

In [175]:
px.scatter_polar(
    train_df, r="avg_radius", theta="avg_angle_deg", color="cable", range_theta=[15, 90]
)

In [185]:
px.box(full_df, y="avg_tension_drop", x="state", color="dataset", facet_col="cable", facet_col_wrap=2, height=800)

In [209]:
t = full_df.groupby(["cable", "state", "dataset"])["elapsed_days"].apply(lambda x: x.max() - x.min()).reset_index()

In [213]:
px.bar(t, x="state", y="elapsed_days", color="dataset", facet_col="cable", facet_col_wrap=2, height=800, barmode="group")

In [199]:
px.line(
    full_df,
    x="avg_radius",
    y="avg_angle_deg",
    facet_col="cable",
    color="dataset",
    width=1200,
    height=500,
)